# 05 — Financial Impact & Retention ROI

## Purpose
A model is only as valuable as the revenue it protects. This notebook translates
the churn findings into a dollar-value business case using a **24-month Customer
Lifetime Value (CLTV) simulation**.

## The Central Argument
The most operationally actionable segment is **Friction Churners**: customers who
churned *and* had an unresolved billing or tech issue in their first 60 days.
These customers are:

1. **Identifiable in real-time** — the friction signal appears within 60 days of acquisition.
2. **Preventable** — resolution rate, not issue occurrence, is the lever.
3. **High-value** — their CLTV is representative of the broader base.

## Key Financial Findings
| Metric                              | Value              |
|-------------------------------------|--------------------|
| Friction-driven churners            | **19,147**         |
| Share of total churn                | **16.7%**          |
| Average 24-month CLTV               | **~$1,695**        |
| Total CLTV at risk (friction only)  | **$32.45 million** |
| CLTV retained at 20% save rate      | **$6.49 million**  |

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/telco_churn_225k_v120.csv")

## Defining Friction Churners

A **friction churner** is any customer who:
- Has `churn == 1` (they left), AND
- Had at least one unresolved service issue in the first 60 days
  (`billing_call_resolved == 0` OR `tech_issue_resolved == 0`)

This segment is distinct from the overall churner population. Most churn is
driven by structural factors (contract type, pricing). Friction churn is driven
by operational failures — and is therefore the most directly addressable.

In [ ]:
friction_mask = (
    (df["churn"] == 1) &
    ((df["billing_call_resolved"] == 0) | (df["tech_issue_resolved"] == 0))
)
friction_churners = df[friction_mask].copy()

total_churners_count      = df["churn"].sum()
friction_churners_count   = len(friction_churners)
pct_friction_of_total     = (friction_churners_count / total_churners_count) * 100
total_cltv_at_risk        = friction_churners["cltv"].sum()
avg_cltv_per_churner      = friction_churners["cltv"].mean()

print("─" * 50)
print("FRICTION CHURN SEGMENT")
print("─" * 50)
print(f"  Total churners              : {total_churners_count:,}")
print(f"  Friction-driven churners    : {friction_churners_count:,}")
print(f"  Friction share of churn     : {pct_friction_of_total:.1f}%")
print(f"  Total 24-month CLTV at risk : ${total_cltv_at_risk:,.0f}")
print(f"  Avg CLTV per churner        : ${avg_cltv_per_churner:,.0f}")

## Save-Rate Simulation — "Rapid Resolution" Program

We model a hypothetical **Rapid Resolution task force**: a dedicated team that
monitors new subscribers for billing and tech issues and intervenes within 48 hours.

The simulation asks: *at each possible save rate, how much CLTV is recovered?*

The **20% save rate** is the strategic target — aggressive enough to be meaningful,
conservative enough to be credible. Industry benchmarks for well-executed early
intervention programs typically land in the 15–25% range.

In [ ]:
save_rates = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
sim_df = pd.DataFrame([{
    "Save Rate (%)":      int(r * 100),
    "Customers Saved":    int(friction_churners_count * r),
    "CLTV Retained ($)":  int(friction_churners_count * r) * avg_cltv_per_churner,
} for r in save_rates])

print(sim_df.to_string(index=False))

## Visualization — CLTV Retained by Save Rate

The green bar (20%) marks the strategic target scenario. The chart is designed
to anchor the conversation: even a conservative 10% save rate recovers over
$3 million in 24-month contribution margin.

In [ ]:
plt.figure(figsize=(10, 6), facecolor="white")
palette = ["#deebf7", "#9ecae1", "#4292c6", "#084594", "#08306b", "#081d58"]
ax = sns.barplot(data=sim_df, x="Save Rate (%)", y="CLTV Retained ($)", palette=palette)
ax.patches[2].set_facecolor("#238b45")  # highlight 20% bar in green

plt.title("Financial Impact: CLTV Retained by Save Rate Scenario",
          color="#08306b", fontsize=14, fontweight="bold", pad=15)
plt.xlabel("Program Save Rate (%)", color="#08306b", fontsize=12)
plt.ylabel("24-Month CLTV Retained ($)", color="#08306b", fontsize=12)
plt.grid(axis="y", alpha=0.3)

for p in ax.patches:
    ax.annotate(f"${p.get_height():,.0f}",
                (p.get_x() + p.get_width() / 2.0, p.get_height()),
                ha="center", va="center", xytext=(0, 9),
                textcoords="offset points", color="#08306b", fontweight="bold")

plt.tight_layout()
plt.savefig("../reports/Financial_Impact_CLTV_Simulation.png", dpi=150)
plt.show()

## Strategic Takeaway — Telecom → Insurance

The friction churn framework transfers directly to personal lines insurance.
The operational analogy:

| Telecom                              | Auto / Home Insurance                           |
|--------------------------------------|-------------------------------------------------|
| Unresolved billing call (60 days)    | Billing setup issue or first-payment failure    |
| Unresolved tech issue (60 days)      | Underwriting delay or first-claim friction      |
| Rapid Resolution task force          | Early-tenure specialist team                    |
| CLTV at risk                         | Premium-at-risk (renewal × margin)              |

**The business case is structurally identical:** identify early friction events,
measure their contribution to non-renewal, and size the investment in resolution
capacity against the premium at risk.

At a 20% save rate, a $6.49M CLTV recovery in telecom becomes an equivalent
dollar argument for a dedicated retention headcount at any carrier running
a similar friction-to-churn analysis on their policyholder base.